In [22]:
import os
import pandas as pd
import numpy as np
import pickle

from tqdm import tqdm
tqdm.pandas()

import mediapipe as mp
import cv2

import matplotlib.pyplot as plt

In [23]:
def extract_frames(video_path, fps=10):
    cap = cv2.VideoCapture(video_path)
    frame_rate = cap.get(cv2.CAP_PROP_FPS)
    frame_interval = int(frame_rate / fps)

    current_frame = 0
    frames_list = []
    while True:
        ret, frame = cap.read()

        if not ret:
            break

        if current_frame % frame_interval == 0:
            frames_list.append(frame)

        current_frame += 1

    cap.release()

    return frames_list

In [24]:
def pose_estimation(frame_list):
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose()

    pose_landmarks_list = []
    cant_estimate_index = []

    for i, frame in enumerate(frame_list):
        results = pose.process(frame)
        if results.pose_landmarks:
            pose_landmarks_list.append(results.pose_landmarks)
        else:
            cant_estimate_index.append(i)
    
    return pose_landmarks_list, cant_estimate_index

In [25]:
def find_max_min_of_longest_consecutive(nums):
    nums = sorted(nums)
    longest_streak = 0
    current_streak = 1
    max_min = (nums[0], nums[0])

    for i in range(1, len(nums)):
        if nums[i] == nums[i - 1] + 1:
            current_streak += 1
        else:
            if current_streak > longest_streak:
                longest_streak = current_streak
                max_min = (nums[i - current_streak], nums[i - 1])
            current_streak = 1

    if current_streak > longest_streak:
        max_min = (nums[-current_streak], nums[-1])

    return max_min

In [26]:
def classify_error(pose_landmarks_list, cant_estimate_index):
    if len(cant_estimate_index) == 0:
        return True
    
    # pose_estimation 실패 비율이 20% 이상인 경우
    elif len(cant_estimate_index) / (len(pose_landmarks_list) + len(cant_estimate_index)) > 0.2:
        return False
        
    # 연속된 pose_estimation 실패가 전체 프레임의 0.1 이상인 경우
    min_max = find_max_min_of_longest_consecutive(cant_estimate_index)
    len_continue = min_max[1] - min_max[0] + 1
    if len_continue / (len(pose_landmarks_list) + len(cant_estimate_index)) > 0.1:
        return False

    else:
        return True

In [14]:
video_base_path = '../Data/tmp/Video/'
video_list = sorted(os.listdir(video_base_path))
save_base_path = '../Data/tmp/Images/'

for video_name in tqdm(video_list):
    save_path = save_base_path + video_name.split('.')[0]
    if os.path.isdir(save_path) == False:
        os.mkdir(save_path)
    else:
        os.rmdir(save_path)
        os.mkdir(save_path)

    video_path = os.path.join(video_base_path, video_name)
    extract_frames(video_path, save_path)

  0%|          | 0/17 [00:00<?, ?it/s]

OSError: [WinError 145] 디렉터리가 비어 있지 않습니다: '../Data/tmp/Images/elbow_high_1'

In [28]:
image_base_path = '../data/tmp/Images/'
save_base_path = '../data/tmp/Keypoints/'

for image_name in os.listdir(image_base_path):
    image_path = os.path.join(image_base_path, image_name)
    print(image_path)
    
    frame_list = []
    for frame_name in os.listdir(image_path):
        frame_path = os.path.join(image_path, frame_name)
        frame = cv2.imread(frame_path)
        if frame is not None:
            frame_list.append(frame)
    
    if frame_list:
        pose_landmarks_list, cant_estimate_index = pose_estimation(frame_list)
        with open(os.path.join(save_base_path, image_name + '.pkl'), 'wb') as f:
            pickle.dump((pose_landmarks_list, cant_estimate_index), f)
        print(f"{image_name} 처리 완료")
    else:
        print(f"{image_name}에서 유효한 프레임을 찾을 수 없습니다.")

../data/tmp/Images/elbow_high_1
elbow_high_1 처리 완료
../data/tmp/Images/elbow_high_2
elbow_high_2 처리 완료
../data/tmp/Images/elbow_low_1
elbow_low_1 처리 완료
../data/tmp/Images/elbow_low_2
elbow_low_2 처리 완료
../data/tmp/Images/elbow_middle_1
elbow_middle_1 처리 완료
../data/tmp/Images/elbow_middle_2
elbow_middle_2 처리 완료
../data/tmp/Images/elbow_middle_3
elbow_middle_3 처리 완료
../data/tmp/Images/normal_high_2
normal_high_2 처리 완료
../data/tmp/Images/normal_high_4
normal_high_4 처리 완료
../data/tmp/Images/normal_low_1
normal_low_1 처리 완료
../data/tmp/Images/normal_low_2
normal_low_2 처리 완료
../data/tmp/Images/shoulder_high_2
shoulder_high_2 처리 완료
../data/tmp/Images/shoulder_high_3
shoulder_high_3 처리 완료
../data/tmp/Images/shoulder_low_1
shoulder_low_1 처리 완료
../data/tmp/Images/shoulder_low_3
shoulder_low_3 처리 완료
../data/tmp/Images/shoulder_middel_3
shoulder_middel_3 처리 완료
../data/tmp/Images/shoulder_middle_3
shoulder_middle_3 처리 완료
